Trains a compact CNN that recognises chess piece glyphs from the actual fonts in your PDF chess book.
The exported TFLite model ships inside the Flutter app and runs fully offline.

**Pipeline**
1. Mount Google Drive and load your chess PDF
2. For pages with games, render them as high-res images
3. Extract text layer with per-character bounding boxes
4. Identify figurine characters (non-ASCII/unusual Unicode)
5. Crop glyphs from rendered images, infer labels from move context
6. Train model on real book fonts
7. Export TFLite model to use in the Flutter app

**Model**
- Input : 32 × 32 grayscale, values in \[0, 1\]
- Output: 6-class softmax — `['K', 'Q', 'R', 'B', 'N', 'P']`
- Size  : < 500 KB (float32 TFLite)

## Step 0 — Mount Google Drive and load PDF

In [ ]:
from google.colab import drive
import os

drive.mount('/content/gdrive')

### Step 0b — Verify PDF file

In [ ]:
# ── EDIT THIS: set your PDF path ────────────────────────────────────────────
PDF_PATH = '/content/gdrive/MyDrive/chess_book.pdf'  # CHANGE THIS to your PDF

# Verify PDF exists
if not os.path.exists(PDF_PATH):
    print(f'❌ PDF not found: {PDF_PATH}')
    print(f'   Available items in /content/gdrive/MyDrive:')
    for item in os.listdir('/content/gdrive/MyDrive')[:20]:
        print(f'     - {item}')
else:
    size_mb = os.path.getsize(PDF_PATH) / (1024*1024)
    print(f'✅ PDF loaded: {PDF_PATH}  ({size_mb:.1f} MB)')

## Step 1 — Install dependencies

In [ ]:
!apt-get install -y poppler-utils tesseract-ocr
!pip install -q pdf2image pdfplumber pillow numpy tensorflow matplotlib chess pytesseract

## Step 2 — Configuration

In [ ]:
# ── Classes ────────────────────────────────────────────────────────────────
CLASS_NAMES = ['K', 'Q', 'R', 'B', 'N', 'P']   # piece letter; P = pawn
IMG_SIZE    = 32   # pixels — model input (32×32 grayscale)

# ── PDF Configuration ──────────────────────────────────────────────────────
# PDF_PATH was set in Step 0b — change it there and re-run Step 0b if needed
# START_PAGE, END_PAGE: pages to extract glyphs from (1-indexed)
# MIN_GLYPH_SAMPLES: minimum distinct glyphs to collect before training (quality filter)

START_PAGE      = 1
END_PAGE        = 20      # extract from first 20 pages
GLYPH_DPI       = 150     # render resolution for glyph extraction
MIN_GLYPH_SAMPLES = 10    # minimum samples per piece to proceed with training

# ── CRITICAL: Does your chess book use FIGURINE notation or PLAIN TEXT notation? ────
#
# FIGURINE notation:  Moves have a special chess symbol (image) for the piece
#                     Example: ♘xf6  (knight symbol rendered as image, followed by xf6)
#                     → We extract the IMAGE and train a classifier
#
# PLAIN TEXT notation: Moves are just standard algebraic text, no special symbols
#                      Example: Nxf6  (N is a regular letter, not a chess font)
#                      → We just parse the text, no classifier needed
#
USE_FIGURINE_NOTATION = True  # ← CHANGE THIS BASED ON YOUR BOOK!
                              # True = figurine notation, extract and classify symbols
                              # False = plain text notation, just parse moves

# ── Augmentation (applied at training time, not collection) ─────────────────
AUGMENT_PER_GLYPH = 40    # variants generated per collected glyph

# ── Training ───────────────────────────────────────────────────────────────
EPOCHS     = 60
BATCH_SIZE = 64
VAL_SPLIT  = 0.15

# ── Output ────────────────────────────────────────────────────────────────
TFLITE_PATH = 'figurine_classifier.tflite'

print('Classes    :', CLASS_NAMES)
print('PDF        :', PDF_PATH)
print('Pages      : %d–%d' % (START_PAGE, END_PAGE))
print('Min samples:', MIN_GLYPH_SAMPLES, 'per piece')
print()
if USE_FIGURINE_NOTATION:
    print('✅ MODE: FIGURINE NOTATION')
    print('   → Will extract figurine glyphs from images')
    print('   → Will train classifier on piece symbols')
else:
    print('✅ MODE: PLAIN TEXT NOTATION')
    print('   → Will parse moves from text only')
    print('   → No figurine classifier needed')

## Step 3 — Extract figurine glyphs from PDF pages (Fixed OCR approach)

**What was wrong:**
The original code tried to OCR the entire word region, which included move data and led to corrupted results like `'93 £7'`, `'Six''`, etc. This corrupted the piece identification.

**What's fixed:**
1. **Single-character OCR mode** — Tesseract `--psm 10` (recognize single character only)
2. **Character whitelist** — restrict output to valid chess piece symbols: `KQRBN♔♕♖♗♘`
3. **Enhanced contrast** — improves OCR accuracy on small rendered glyphs
4. **Proper fallback** — if OCR fails on symbol, check if text contains a move (file+rank) and assume pawn
5. **Real image extraction** — crop from rendered PDF using word bounding boxes
6. **Ground-truth labeling** — piece type comes from OCR symbol detection

**Result:** extraction_results contains:
- `crop_image`: actual figurine glyph from PDF
- `inferred_piece`: detected piece type (K, Q, R, B, N, or P)
- `ocr_symbol`: what OCR returned (for debugging)


In [ ]:
import pdfplumber
import numpy as np
from PIL import Image, ImageEnhance
from pdf2image import convert_from_path
import pytesseract
from collections import defaultdict
import re

# ── Helper: render page to high-res image ──────────────────────────────────
def render_page(pdf_path, page_num, dpi=150):
    """page_num is 0-indexed"""
    images = convert_from_path(pdf_path, first_page=page_num+1, last_page=page_num+1, dpi=dpi)
    return images[0] if images else None

# ── Filter: is this word a chess move? ──────────────────────────────────────
def looks_like_chess_move(text):
    """
    Check if text looks like a chess move.
    
    Valid patterns (both notations):
      Kd4, Nxf6, e4, dxe5, Rxc3, etc.
    
    Pattern:
      ^[KQRBN]?      — optional piece letter
      [a-h]          — file
      x?             — optional capture
      [a-h]          — destination file
      [1-8]          — destination rank
      [+#=]?         — optional check/mate/promotion
      [!?]*          — optional annotations
    """
    clean = text.strip().upper()
    
    # Must contain file and rank
    has_file = bool(re.search(r'[A-H]', clean))
    has_rank = bool(re.search(r'[1-8]', clean))
    if not (has_file and has_rank):
        return False
    
    # Match move pattern
    move_pattern = r'^[KQRBN]?[A-H]x?[A-H][1-8][+#=]?[!?]*$'
    if not re.match(move_pattern, clean):
        return False
    
    # Reject very long words
    if len(clean) > 12:
        return False
    
    return True

# ── OCR figurine symbol (for FIGURINE notation only) ──────────────────────
def ocr_symbol_region(rendered_image, bbox, target_dpi=GLYPH_DPI):
    """OCR a small region to detect figurine symbol."""
    scale = target_dpi / 72.0
    x0 = max(0, int(bbox[0] * scale))
    y0 = max(0, int(bbox[1] * scale))
    x1 = min(rendered_image.width, int(bbox[2] * scale))
    y1 = min(rendered_image.height, int(bbox[3] * scale))
    
    if x1 <= x0 or y1 <= y0:
        return None
    
    crop = rendered_image.crop((x0, y0, x1, y1))
    if crop.mode != 'L':
        crop = crop.convert('L')
    
    enhancer = ImageEnhance.Contrast(crop)
    crop = enhancer.enhance(2.5)
    
    try:
        text = pytesseract.image_to_string(
            crop, 
            config='--psm 10 -c tessedit_char_whitelist=KQRBN♔♕♖♗♘♚♛♜♝♞'
        ).strip()
        return text if text else None
    except:
        return None

# ── Infer piece from move text (PLAIN TEXT notation) ──────────────────────
def infer_piece_from_text(text):
    """
    For plain text notation: extract piece from text directly.
    Kd4 → K, Nxf6 → N, e4 → P, etc.
    """
    clean = text.strip().upper()
    first = clean[0] if clean else None
    
    if first in 'KQRBN':
        return first
    if first in 'ABCDEFGH':
        return 'P'  # pawn
    
    return None

# ── Infer piece from figurine symbol or text (FIGURINE notation) ──────────
def infer_piece_from_figurine(text, ocr_symbol):
    """
    For figurine notation: piece comes from OCR symbol or text fallback.
    If text has piece letter → use it
    If text has no piece letter but OCR found symbol → use OCR
    If neither → pawn
    """
    clean = text.strip().upper()
    first = clean[0] if clean else None
    
    # Case 1: piece letter in text
    if first in 'KQRBN':
        return first
    
    # Case 2: piece letter in figurine image (OCR)
    if first in 'ABCDEFGH' and ocr_symbol:
        ocr_clean = ocr_symbol.strip().upper()
        if ocr_clean in 'KQRBN':
            return ocr_clean
        symbol_map = {
            '♔': 'K', '♕': 'Q', '♖': 'R', '♗': 'B', '♘': 'N',
            '♚': 'K', '♛': 'Q', '♜': 'R', '♝': 'B', '♞': 'N',
        }
        if ocr_clean in symbol_map:
            return symbol_map[ocr_clean]
    
    # Case 3: no piece found → pawn
    if first in 'ABCDEFGH':
        return 'P'
    
    return None

# ── Main extraction ─────────────────────────────────────────────────────────
extraction_results = []

if USE_FIGURINE_NOTATION:
    print(f'Extracting FIGURINE notation from pages {START_PAGE} to {END_PAGE}...\n')
else:
    print(f'Extracting PLAIN TEXT notation from pages {START_PAGE} to {END_PAGE}...\n')

with pdfplumber.open(PDF_PATH) as pdf:
    pdf_page_count = len(pdf.pages)
    if END_PAGE > pdf_page_count:
        END_PAGE = pdf_page_count
        print(f'  (PDF has {pdf_page_count} pages, adjusted END_PAGE)\n')
    
    for page_idx in range(START_PAGE - 1, min(END_PAGE, pdf_page_count)):
        pdf_page = pdf.pages[page_idx]
        print(f'Processing page {page_idx + 1}...')
        
        # Render page (needed for figurine extraction)
        page_image = None
        if USE_FIGURINE_NOTATION:
            page_image = render_page(PDF_PATH, page_idx, dpi=GLYPH_DPI)
            if page_image is None:
                print(f'  ⚠ Could not render page')
                continue
        
        # Extract words
        try:
            words = pdf_page.extract_words()
        except:
            print(f'  ⚠ Could not extract words')
            continue
        
        if not words:
            print(f'  ⚠ No words found')
            continue
        
        page_count = 0
        for word in words:
            text = word.get('text', '')
            
            # Filter: only process chess moves
            if not looks_like_chess_move(text):
                continue
            
            bbox = (word['x0'], word['top'], word['x1'], word['bottom'])
            
            # Infer piece type based on notation mode
            ocr_symbol = None
            if USE_FIGURINE_NOTATION:
                ocr_symbol = ocr_symbol_region(page_image, bbox, target_dpi=GLYPH_DPI)
                piece = infer_piece_from_figurine(text, ocr_symbol)
            else:
                piece = infer_piece_from_text(text)
            
            if not piece:
                continue
            
            # Extract image crop (for both modes, for consistency)
            scale = GLYPH_DPI / 72.0 if page_image else 1.0
            x0 = max(0, int(bbox[0] * scale))
            y0 = max(0, int(bbox[1] * scale))
            x1 = min(page_image.width if page_image else 9999, int(bbox[2] * scale))
            y1 = min(page_image.height if page_image else 9999, int(bbox[3] * scale))
            
            if x1 - x0 >= 8 and y1 - y0 >= 8:
                if page_image:
                    crop = page_image.crop((x0, y0, x1, y1))
                    if crop.mode != 'L':
                        crop = crop.convert('L')
                else:
                    crop = None
                
                extraction_results.append({
                    'page': page_idx + 1,
                    'move_text': text,
                    'ocr_symbol': ocr_symbol,
                    'inferred_piece': piece,
                    'crop_image': crop,
                    'crop_width': x1 - x0,
                    'crop_height': y1 - y0,
                })
                
                ocr_str = f"OCR: {repr(ocr_symbol):8}" if USE_FIGURINE_NOTATION else "OCR: N/A    "
                print(f'  Move: {text:12} | {ocr_str} | Piece: {piece}  ({x1-x0}×{y1-y0} px)')
                page_count += 1
        
        print(f'  ✅ Extracted {page_count} moves\n')

print(f'📊 Extraction Summary:')
print(f'  Total moves extracted: {len(extraction_results)}')
print(f'  Mode: {"FIGURINE notation" if USE_FIGURINE_NOTATION else "PLAIN TEXT notation"}\n')

piece_counts = defaultdict(int)
for result in extraction_results:
    piece_counts[result['inferred_piece']] += 1

print('Moves by piece type:')
for piece in ['K', 'Q', 'R', 'B', 'N', 'P']:
    count = piece_counts[piece]
    status = '✅' if count >= MIN_GLYPH_SAMPLES else '⚠'
    if count > 0:
        print(f'  {status} {piece}: {count}')

if len(extraction_results) == 0:
    print('\n❌ No moves extracted!')
    print('   Check: PDF path, page range, move notation')


## Step 3b — Extract glyph samples and build training dataset

In [ ]:
# Step 3a is no longer needed!
# 
# The improved Step 3 now:
#   1. Extracts image crops from the rendered PDF
#   2. Infers piece types directly from move TEXT (not OCR)
#   3. Stores both the image and the correct label
#
# Result: extraction_results contains all glyphs with ground-truth labels
# ready for training.

print('✅ Step 3a skipped — labeling is done via move notation text.')
print('✅ Proceed to augmentation and training.')


In [ ]:
import numpy as np
from PIL import Image, ImageOps, ImageFilter
import random

def augment_glyph(pil_img, n, target_size=32):
    """Generate n augmented versions of a glyph image for training."""
    results = []
    
    # Resize to target size
    if pil_img.size[0] == 0 or pil_img.size[1] == 0:
        return []
    
    base = pil_img.resize((target_size, target_size), Image.LANCZOS)
    
    for _ in range(n):
        img = base.copy()
        
        # Random rotation ±12°
        angle = random.uniform(-12, 12)
        img = img.rotate(angle, resample=Image.BICUBIC, fillcolor=255)
        
        # Random scale 0.80 – 1.20
        scale  = random.uniform(0.80, 1.20)
        new_sz = max(4, int(target_size * scale))
        img    = img.resize((new_sz, new_sz), Image.LANCZOS)
        canvas = Image.new('L', (target_size, target_size), 255)
        off    = (target_size - new_sz) // 2
        paste_box = (max(0, off), max(0, off),
                     min(target_size, new_sz + off), min(target_size, new_sz + off))
        crop_box = (max(0, -off), max(0, -off),
                    min(new_sz, target_size - off), min(new_sz, target_size - off))
        img_crop = img.crop(crop_box)
        canvas.paste(img_crop, paste_box)
        img = canvas
        
        # Random horizontal flip
        if random.random() > 0.5:
            img = ImageOps.mirror(img)
        
        # Occasional subtle blur
        if random.random() > 0.8:
            img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(0.2, 0.6)))
        
        # Convert to float32 in [0, 1]
        arr = np.array(img, dtype=np.float32) / 255.0
        
        # Random noise
        arr += np.random.normal(0, 0.03, arr.shape)
        
        # Random brightness
        brightness = random.uniform(0.85, 1.15)
        arr = arr * brightness + random.uniform(-0.04, 0.04)
        arr = np.clip(arr, 0.0, 1.0)
        
        results.append(arr)
    
    return results

# ── Build training dataset from extracted glyphs ────────────────────────────
print(f'Building training dataset from {len(extraction_results)} extracted glyphs...\n')

if not extraction_results:
    print('❌ No glyphs to process. Run Step 3 first.')
else:
    X, y = [], []
    samples_per_piece = defaultdict(int)
    
    for result in extraction_results:
        crop_image = result['crop_image']
        piece = result['inferred_piece']
        piece_idx = CLASS_NAMES.index(piece)
        
        # Augment each extracted glyph
        augmented = augment_glyph(crop_image, AUGMENT_PER_GLYPH)
        for aug_array in augmented:
            X.append(aug_array)
            y.append(piece_idx)
            samples_per_piece[piece] += 1
    
    if len(X) > 0:
        X = np.array(X)[..., np.newaxis]  # (N, 32, 32, 1)
        y = np.array(y)
        
        print(f'✅ Generated {len(X)} training samples from {len(extraction_results)} glyphs:')
        for piece in CLASS_NAMES:
            count = samples_per_piece.get(piece, 0)
            status = '✅' if count >= MIN_GLYPH_SAMPLES else '⚠'
            print(f'   {status} {piece}: {count}')
        
        if len(X) < 100:
            print(f'\n⚠ Warning: Only {len(X)} samples. Consider extending END_PAGE.')
    else:
        print('❌ No augmented samples generated.')


## Step 3a — OCR-Based Symbol Detection

**How it works:**

Each move in the PDF has this structure:
```
[FIGURINE_SYMBOL?] [MOVE_DATA]
```

Where:
- `[FIGURINE_SYMBOL]` = optional glyph (K, Q, R, B, N) or Unicode chess symbol (♔, ♕, ♖, ♗, ♘)
- `[MOVE_DATA]` = destination square, captures, checks (e.g., "xf7+", "d4", "Nxe5")

**Detection strategy:**
1. Extract word bounding box from PDF
2. OCR the region with Tesseract in **single-character mode** (`--psm 10`)
3. Restrict OCR output to chess pieces: `KQRBN♔♕♖♗♘`
4. Map OCR'd symbol to piece type:
   - `K` / `♔` / `♚` → King
   - `Q` / `♕` / `♛` → Queen
   - `R` / `♖` / `♜` → Rook
   - `B` / `♗` / `♝` → Bishop
   - `N` / `♘` / `♞` → Knight
   - No symbol + contains file+rank (e.g., "e4") → Pawn

5. Extract the actual image crop from the rendered PDF
6. Label it with the inferred piece type from step 4

**Key improvement:** Tesseract single-character mode (`--psm 10`) + character whitelist = cleaner OCR on small glyphs


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(6, 10, figsize=(14, 9))
for cls_idx, cls_name in enumerate(CLASS_NAMES):
    samples = X[y == cls_idx][:10]
    for col, img in enumerate(samples):
        ax = axes[cls_idx][col]
        ax.imshow(img.squeeze(), cmap='gray', vmin=0, vmax=1)
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(cls_name, fontsize=11, rotation=0, labelpad=20, va='center')
plt.suptitle('Sample training images (10 per class)', fontsize=12)
plt.tight_layout()
plt.show()

## Step 4 — Train CNN on extracted glyphs

In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

# Check if we have training data
if 'X' not in locals() or len(X) == 0:
    print('❌ No training data. Run the glyph extraction step above first.')
else:
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=VAL_SPLIT, stratify=y, random_state=42
    )
    print(f'Train: {len(X_train)}  Val: {len(X_val)}')
    
    # ── Model ──────────────────────────────────────────────────────────────────
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
    
        tf.keras.layers.Conv2D(32, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPool2D(2),
    
        tf.keras.layers.Conv2D(64, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.MaxPool2D(2),
    
        tf.keras.layers.Conv2D(128, 3, padding='same', activation='relu'),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.GlobalAveragePooling2D(),
    
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dropout(0.4),
        tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax'),
    ], name='figurine_classifier')
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    model.summary()
    
    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True,
                                         monitor='val_accuracy'),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=5, min_lr=1e-5,
                                             monitor='val_accuracy'),
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=1,
    )
    
    val_acc = max(history.history['val_accuracy'])
    print(f'\n✅ Best val accuracy: {val_acc:.1%}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['accuracy'],     label='train')
ax1.plot(history.history['val_accuracy'], label='val')
ax1.set_title('Accuracy'); ax1.legend(); ax1.set_xlabel('epoch')
ax2.plot(history.history['loss'],     label='train')
ax2.plot(history.history['val_loss'], label='val')
ax2.set_title('Loss'); ax2.legend(); ax2.set_xlabel('epoch')
plt.tight_layout(); plt.show()

# Confusion matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
cm = confusion_matrix(y_val, y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(ax=ax, colorbar=False)
plt.title('Validation confusion matrix')
plt.tight_layout(); plt.show()

## Step 5 — Export TFLite model

In [ ]:
if 'model' not in locals():
    print('❌ Model not trained. Run the training step first.')
else:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    # float32 — compatible with all TFLite runtimes, no calibration needed.
    tflite_model = converter.convert()
    
    with open(TFLITE_PATH, 'wb') as f:
        f.write(tflite_model)
    
    size_kb = os.path.getsize(TFLITE_PATH) / 1024
    print(f'✅ Saved: {TFLITE_PATH}  ({size_kb:.0f} KB)')
    
    # ── Sanity-check on the TFLite model ──────────────────────────────────
    interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
    interp.allocate_tensors()
    inp = interp.get_input_details()[0]
    out = interp.get_output_details()[0]
    print(f'   Input : {inp["shape"]}  dtype={inp["dtype"].__name__}')
    print(f'   Output: {out["shape"]}  dtype={out["dtype"].__name__}')
    
    # Run one test image per class (if available).
    print('\nClass-level spot check:')
    for cls_idx, cls_name in enumerate(CLASS_NAMES):
        samples = X_val[y_val == cls_idx]
        if len(samples) > 0:
            sample = samples[0:1].astype(np.float32)
            interp.set_tensor(inp['index'], sample)
            interp.invoke()
            probs = interp.get_tensor(out['index'])[0]
            pred  = CLASS_NAMES[np.argmax(probs)]
            conf  = probs.max()
            ok    = '✅' if pred == cls_name else '⚠'
            print(f'  {ok} true={cls_name}  pred={pred}  conf={conf:.1%}')
        else:
            print(f'  ⚠ {cls_name}: no validation samples')

In [ ]:
from google.colab import files

if os.path.exists(TFLITE_PATH):
    files.download(TFLITE_PATH)
    print(f'✅ Downloaded {TFLITE_PATH}')
else:
    print(f'❌ Model file not found: {TFLITE_PATH}. Run export step first.')

## Flutter integration

1. **Before running this notebook:**
   - Upload your chess PDF to Google Drive (the one you want to read with the app)
   - Edit `PDF_PATH` in Step 1 to point to your PDF
   - Run Step 2 to extract glyphs and see which characters are piece symbols
   - Create a `GLYPH_MAPPING` in Step 3a mapping each character to its piece letter

2. **After training:**
   - Download `figurine_classifier.tflite` from the notebook
   - Copy it to `assets/models/figurine_classifier.tflite` in your Flutter project
   - Declare the asset in `pubspec.yaml` (already done)
   - Rebuild and run the app

3. **In the app:**
   - The `FigurineClassifier` loads the model at startup
   - For each PDF page: renders it, extracts text with bboxes
   - For non-ASCII characters (piece glyphs), crops from the rendered image
   - Runs this model to get confidence scores
   - Builds `fontMap` from char → piece prediction
   - Passes `fontMap` to `MoveParser` for accurate move parsing

4. **Model expectations:**
   - Input: Float32List of length 1024 (32×32 pixels, values in \[0,1\])
   - Output: Float32List of length 6 (probabilities for K, Q, R, B, N, P)
   - Use `argmax(output)` to get predicted piece class